In [1]:
# import
import torch
import torchvision
import torchvision.transforms as transforms
import torchvision.models as models
import torch.nn as nn
import torch.optim as optim

In [2]:
# Define data transforms
transform = transforms.Compose([
    transforms.Resize(224),  # ResNet models usually expect 224x224 input
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

In [3]:
trainset = torchvision.datasets.CIFAR100(root='./data', train= True, download=True, transform = transform)
testset = torchvision.datasets.CIFAR100(root='./data', train= False, download=True, transform = transform)

100%|██████████| 169001437/169001437 [00:03<00:00, 48313911.82it/s]


Extracting ./data/cifar-100-python.tar.gz to ./data
Files already downloaded and verified


In [4]:
trainloader = torch.utils.data.DataLoader(trainset, batch_size=64, shuffle=True)
testloader = torch.utils.data.DataLoader(testset, batch_size = 64, shuffle=False)

In [5]:
data = iter(trainloader)
image, label = next(data)
print(label)
print(image.shape)

tensor([30, 51, 65, 22, 28, 41, 63, 51, 58, 56, 24, 36, 60, 37, 67, 92, 45, 99,
        82, 11, 14, 11, 78,  4,  5,  4, 24,  1, 91, 33, 12, 76, 38, 47, 87, 92,
        91, 89, 59, 50,  8, 68, 49, 18, 31, 77, 60, 48, 97, 23, 83, 33,  8, 45,
        78, 75, 94, 46, 23, 58, 52, 94,  3, 42])
torch.Size([64, 3, 224, 224])


In [6]:
model = models.resnet50(weights= models.ResNet50_Weights.DEFAULT)

Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /root/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth
100%|██████████| 97.8M/97.8M [00:00<00:00, 134MB/s]


In [7]:
# Modify the final layer to fit CIFAR-100 (100 output classes instead of 1000 for ImageNet)
num_ftrs = model.fc.in_features
model.fc = nn.Linear(num_ftrs, 100)

In [8]:
# Use GPU if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

In [9]:
criteria= nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

In [10]:
for epoch in range(10):
  running_loss = 0
  for i, data in enumerate(trainloader, 0):
    images, labels = data
    images, labels = images.to(device), labels.to(device)
    optimizer.zero_grad()
    output = model(images)
    loss = criteria(output, labels)
    loss.backward()
    optimizer.step()

    # print statistics
    running_loss += loss.item()
    if i % 100 == 99:    # print every 10 mini-batches
        print(f'[{epoch + 1}, {i + 1}] loss: {running_loss / 10:.3f}')
        running_loss = 0.0

print('Finished Training')

[1, 100] loss: 31.169
[1, 200] loss: 20.292
[1, 300] loss: 17.408
[1, 400] loss: 15.995
[1, 500] loss: 14.609
[1, 600] loss: 13.320
[1, 700] loss: 13.171
[2, 100] loss: 9.458
[2, 200] loss: 9.687
[2, 300] loss: 9.520
[2, 400] loss: 9.796
[2, 500] loss: 9.436
[2, 600] loss: 9.340
[2, 700] loss: 9.479
[3, 100] loss: 5.906
[3, 200] loss: 6.087
[3, 300] loss: 6.546
[3, 400] loss: 6.663
[3, 500] loss: 6.585
[3, 600] loss: 6.946
[3, 700] loss: 7.119
[4, 100] loss: 3.995
[4, 200] loss: 4.107
[4, 300] loss: 4.428
[4, 400] loss: 4.646
[4, 500] loss: 4.735
[4, 600] loss: 4.830
[4, 700] loss: 5.144
[5, 100] loss: 2.634
[5, 200] loss: 2.639
[5, 300] loss: 2.750
[5, 400] loss: 2.924
[5, 500] loss: 3.191
[5, 600] loss: 3.452
[5, 700] loss: 3.846
[6, 100] loss: 1.865
[6, 200] loss: 1.804
[6, 300] loss: 1.970
[6, 400] loss: 2.106
[6, 500] loss: 2.377
[6, 600] loss: 2.647
[6, 700] loss: 2.568
[7, 100] loss: 1.608
[7, 200] loss: 1.393
[7, 300] loss: 1.561
[7, 400] loss: 1.565
[7, 500] loss: 1.962
[7, 60

In [11]:
correct = 0
total = 0
with torch.no_grad():
  for data in testloader:
    images, labels = data
    images, labels = images.to(device), labels.to(device)
    outputs = model(images)
    _, predicted = torch.max(outputs, 1)
    total += labels.size(0)
    correct += (predicted == labels).sum().item()

print(f'Accuracy of the network on the 10000 test images: {100 * correct // total} %')

Accuracy of the network on the 10000 test images: 73 %


In [12]:

print(device)

cuda
